# 作业3：卷积神经网络基础、经典CNN、BN、残差、图像增广、目标检测

## 2 卷积和池化层
### 2.1 理论计算题
输入：$C_{in}=3,H=32,W=32$；卷积核：$K=5×5$，输出通道$C_{out}=16$，$P=2,S=2$
特征图尺寸公式：$H_{out}=\lfloor \frac{H+2P-K}{S}\rfloor+1$
$$H_{out}=W_{out}=\lfloor\frac{32+4-5}{2}\rfloor+1=16$$
1. 输出尺寸：$\boldsymbol{16×16×16}$(通道×高×宽)
2. 单个输出像素点乘次数：输入通道3，$3×5×5=\boldsymbol{75}$次乘法

### 2.2 手动实现MaxPool2d(Numpy)

In [1]:
import numpy as np

def max_pool2d(inputs, kernel_size, stride, padding):
    # inputs: [N,C,H,W]
    N,C,H,W = inputs.shape
    kh, kw = kernel_size if isinstance(kernel_size,tuple) else (kernel_size,kernel_size)
    ph, pw = padding if isinstance(padding,tuple) else (padding,padding)
    sh, sw = stride if isinstance(stride,tuple) else (stride,stride)
    # padding
    pad_in = np.pad(inputs, ((0,0),(0,0),(ph,ph),(pw,pw)), mode='constant')
    H_pad, W_pad = pad_in.shape[2], pad_in.shape[3]
    out_h = (H_pad - kh)//sh + 1
    out_w = (W_pad - kw)//sw + 1
    out = np.zeros((N,C,out_h,out_w))
    for i in range(out_h):
        for j in range(out_w):
            h_start = i*sh
            h_end = h_start + kh
            w_start = j*sw
            w_end = w_start + kw
            pool_region = pad_in[:,:,h_start:h_end,w_start:w_end]
            out[:,:,i,j] = np.max(pool_region, axis=(-2,-1))
    return out

# 测试
if __name__ == '__main__':
    test_x = np.random.rand(1,3,32,32)
    pool_out = max_pool2d(test_x,kernel_size=2,stride=2,padding=0)
    print("池化输出shape:",pool_out.shape)

池化输出shape: (1, 3, 16, 16)


## 3 LeNet、AlexNet、VGG、NiN
### 3.1 理论计算（输入输出通道均为C，无偏置）
1. 单层5×5卷积参数量：$C×C×5×5=\boldsymbol{25C^2}$
2. 两层串联3×3卷积：每层$9C^2$，合计$\boldsymbol{18C^2}$

### 3.2 编程：定义NiN Block

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Sequential):
    def __init__(self,in_channels,out_channels,kernel_size,stride,padding):
        super().__init__(
            nn.Conv2d(in_channels,out_channels,kernel_size,stride,padding),
            nn.ReLU(),
            nn.Conv2d(out_channels,out_channels,kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels,out_channels,kernel_size=1),
            nn.ReLU()
        )
#测试
if __name__ == '__main__':
    block = NiNBlock(3,16,kernel_size=3,stride=1,padding=1)
    x = torch.randn(1,3,32,32)
    y = block(x)
    print("NiN块输出shape:",y.shape)

NiN块输出shape: torch.Size([1, 16, 32, 32])


## 4 Inception、BN、残差网络
### 4.1 BN理论计算
$x=[2,4,6,8],\gamma=2,\beta=1,\epsilon=0$
$\mu=\frac{2+4+6+8}{4}=5,\sigma^2=\frac{(9+1+1+9)}{4}=5$
$$y_i=\gamma\frac{x_i-\mu}{\sqrt{\sigma^2}}+\beta$$
$y_1=1-\frac{6}{\sqrt5},y_2=1-\frac2{\sqrt5},y_3=1+\frac2{\sqrt5},y_4=1+\frac6{\sqrt5}$

### 4.2 编程：自定义Residual残差块

In [3]:
import torch.nn as nn
class Residual(nn.Module):
    def __init__(self,in_channels,out_channels,use_1x1conv=False,stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1,stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride)
    def forward(self,x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        return torch.relu(y+x)
#测试
if __name__ == '__main__':
    blk = Residual(3,64,use_1x1conv=True,stride=2)
    x = torch.rand(1,3,32,32)
    print("残差块输出shape:",blk(x).shape)

残差块输出shape: torch.Size([1, 64, 16, 16])


## 5 图像增广、微调与风格迁移
### 5.1 微调理论
1. 预训练底层已经提取通用基础视觉特征，参数成熟，小学习率微调；新分类层随机初始化、需要快速适配新数据集，使用大学习率。冻结底层可避免破坏通用特征。
2. 数据集小且同源：冻结全部特征主干，仅训练最后分类层，完全固定预训练权重，仅优化输出层参数，避免过拟合。

### 5.2 图像增广Pipeline

In [4]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=224,scale=(0.08,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0),
    transforms.ToTensor()
])
#测试
from PIL import Image
img = Image.new('RGB',(256,256))
t = train_transform(img)
print("增强后张量shape:",t.shape)

增强后张量shape: torch.Size([3, 224, 224])


## 6 目标检测与训练技巧
### 6.1 IoU计算
A=[10,10,50,50],B=[30,30,70,70]
交集：(30,30)~(50,50)，面积$20×20=400$
$S_A=40×40=1600,S_B=1600$，并集$=1600+1600-400=2800$
$IoU=\frac{400}{2800}=\boldsymbol{\frac17≈0.142857}$

### 6.2 标签平滑交叉熵实现，$\epsilon=0.1$

In [5]:
import torch
import torch.nn.functional as F

def label_smooth_ce(logits,labels,num_classes,eps=0.1):
    # logits:[N,C],labels:[N]
    N = labels.shape[0]
    smooth_target = torch.full((N,num_classes), fill_value=eps/(num_classes-1), device=logits.device)
    smooth_target.scatter_(dim=1,index=labels.unsqueeze(1),value=1-eps)
    log_prob = F.log_softmax(logits,dim=-1)
    loss = -(smooth_target * log_prob).sum(dim=-1).mean()
    return loss
#测试
if __name__ == '__main__':
    pred = torch.randn(8,10)
    lab = torch.randint(0,10,(8,))
    loss = label_smooth_ce(pred,lab,num_classes=10,eps=0.1)
    print("标签平滑损失值:",loss.item())

标签平滑损失值: 3.356759548187256
